# Titanic Survival Prediction: A Machine Learning Tutorial

This notebook walks through a complete machine learning workflow to predict passenger survival on the Titanic. We will use a **Support Vector Classifier (SVC)** and focus on best practices like data cleaning, pipeline building, and hyperparameter tuning with Grid Search.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

## 1. Data Loading and Initial Exploration

We start by loading our raw data. Exploratory Data Analysis (EDA) is crucial because it helps us identify missing values and understand the data types (numerical vs. categorical) we need to handle later.

In [3]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("titanic_data_updated.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
146,147,yes,third,"Andersson, Mr. August Edvard (""Wennerstrom"")",male,27.0,0,0,350043,7.7958,NaN,S
514,515,no,third,"Coleff, Mr. Satio",male,24.0,0,0,349209,7.4958,NaN,S
657,658,no,third,"Bourke, Mrs. John (Catherine)",female,32.0,1,1,364849,15.5000,NaN,Q
436,437,no,third,"Ford, Miss. Doolina Margaret ""Daisy""",female,21.0,2,2,W./C. 6608,34.3750,NaN,S
359,360,yes,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,NaN,Q


In [4]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [5]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
213,214,no,second,"Givard, Mr. Hans Kristensen",male,30.0,0,0,250646,13.0000,Missing,S,1,M
511,512,no,third,"Webber, Mr. James",male,NaN,0,0,SOTON/OQ 3101316,8.0500,Missing,S,1,M
50,51,no,third,"Panula, Master. Juha Niilo",male,7.0,4,1,3101295,39.6875,Missing,S,6,M
693,694,no,third,"Saad, Mr. Khalil",male,25.0,0,0,2672,7.2250,Missing,C,1,M
452,453,no,first,"Foreman, Mr. Benjamin Laventall",male,30.0,0,0,113051,27.7500,C111,C,1,C


In [6]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [7]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

In [8]:
# Split data: 80% for training the model and 20% for testing its accuracy
# 'stratify=y' ensures both sets have a similar percentage of survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [10]:
y_train

692    yes
481     no
527     no
855    yes
801    yes
      ... 
359    yes
258    yes
736     no
462     no
507    yes
Name: Survived, Length: 712, dtype: object

In [11]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

C:\Users\User\AppData\Local\Temp\ipykernel_10260\987313337.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


In [12]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [13]:
# categories goes one after another via stricly following the order of the input
categories = [['third','second','first']]

In [14]:
# pipeline

#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [16]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [17]:
SVC_model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',SVC())
    ]
)

In [18]:
from sklearn.model_selection import GridSearchCV

In [19]:
grid_param =[ {
    "model__kernel" : ['linear'],
    "model__C" :[0.01 , 0.1 , 1 , 10 , 50 , 100]
},
  {
      "model__kernel" : ['rbf'] ,
      "model__C" :[0.01 , 0.1 ,1,100],
      "model__gamma" :[0.01,0.1,5,10,'scale','auto']
  },
    {
        "model__kernel": ['poly'],
        "model__C" :[0.01 , 0.1 , 1 ,100],
        "model__degree" : [2,3]
    }

]

In [20]:
best_SVC_model = GridSearchCV(estimator = SVC_model , param_grid = grid_param , cv = 5)

In [21]:
best_SVC_model.fit(X_train,y_train)

E:\anaconda\anaconda_install\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
E:\anaconda\anaconda_install\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
E:\anaconda\anaconda_install\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
E:\anaconda\anaconda_install\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
E:\anaconda\anaconda_install\Lib\site-packages\sklearn\preprocessing\_encoders.py:24

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('pipeline_1',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age']),
                                                                        ('pipeline_2',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Fare',
                                                                          'Family_Size']),
                                                                        ('pipeline_3',
                                                                         Pipeline(steps=[('im...
                                                                                          OrdinalEncoder(categories=[['third',
                                                                                                                      'second',
                                                                                                                      'first']])),
                                                                                         ('scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Pclass'])])),
                                       ('model', SVC())]),
             param_grid=[{'model__C': [0.01, 0.1, 1, 10, 50, 100],
                          'model__kernel': ['linear']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__gamma': [0.01, 0.1, 5, 10, 'scale', 'auto'],
                          'model__kernel': ['rbf']},
                         {'model__C': [0.01, 0.1, 1, 100],
                          'model__degree': [2, 3], 'model__kernel': ['poly']}])